[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/langchain-ai/langchain-academy/blob/main/module-0/basics.ipynb) [![Open in LangChain Academy](https://cdn.prod.website-files.com/65b8cd72835ceeacd4449a53/66e9eba12c7b7688aa3dbb5e_LCA-badge-green.svg)](https://academy.langchain.com/courses/take/intro-to-langgraph/lessons/56295530-getting-set-up-video-guide)

# LangChain Academy

Welcome to LangChain Academy! 

## Context

At LangChain, we aim to make it easy to build LLM applications. One type of LLM application you can build is an agent. There’s a lot of excitement around building agents because they can automate a wide range of tasks that were previously impossible. 

In practice though, it is incredibly difficult to build systems that reliably execute on these tasks. As we’ve worked with our users to put agents into production, we’ve learned that more control is often necessary. You might need an agent to always call a specific tool first or use different prompts based on its state. 

To tackle this problem, we’ve built [LangGraph](https://langchain-ai.github.io/langgraph/) — a framework for building agent and multi-agent applications. Separate from the LangChain package, LangGraph’s core design philosophy is to help developers add better precision and control into agent workflows, suitable for the complexity of real-world systems.

## Course Structure

The course is structured as a set of modules, with each module focused on a particular theme related to LangGraph. You will see a folder for each module, which contains a series of notebooks. A video will accompany each notebook to help walk through the concepts, but the notebooks are also stand-alone, meaning that they contain explanations and can be viewed independently of the videos. Each module folder also contains a `studio` folder, which contains a set of graphs that can be loaded into [LangGraph Studio](https://github.com/langchain-ai/langgraph-studio), our IDE for building LangGraph applications.

## Setup

Before you begin, please follow the instructions in the `README` to create an environment and install dependencies.

## Chat models

In this course, we'll be using [Chat Models](https://python.langchain.com/v0.2/docs/concepts/#chat-models), which do a few things take a sequence of messages as inputs and return chat messages as outputs. LangChain does not host any Chat Models, rather we rely on third party integrations. [Here](https://python.langchain.com/v0.2/docs/integrations/chat/) is a list of 3rd party chat model integrations within LangChain! By default, the course will use [ChatOpenAI](https://python.langchain.com/v0.2/docs/integrations/chat/openai/) because it is both popular and performant. As noted, please ensure that you have an `OPENAI_API_KEY`.

Let's check that your `OPENAI_API_KEY` is set and, if not, you will be asked to enter it.

In [1]:
%%capture --no-stderr
%pip install --quiet -U langchain_openai langchain_core langchain_community tavily-python

In [4]:
# Load environment variables from .env file in the parent directory
from pathlib import Path
from dotenv import load_dotenv
import os

# Try multiple possible locations for .env file
possible_paths = [
    Path.cwd().parent / '.env',  # Parent directory
    Path.cwd() / '.env',         # Current directory
    Path('.env')                 # Relative path
]

for path in possible_paths:
    if path.exists():
        load_dotenv(dotenv_path=path)
        print(f"Loaded .env from: {path}")
        break
else:
    print("No .env file found in expected locations")

print("Environment variables loaded:")
print("OPENAI_API_KEY:", "[LOADED]" if os.getenv("OPENAI_API_KEY") else "Not found")
print("TAVILY_API_KEY:", "[LOADED]" if os.getenv("TAVILY_API_KEY") else "Not found")

Loaded .env from: /Users/kazhian/Dev/agentic_ai/.env
Environment variables loaded:
OPENAI_API_KEY: [LOADED]
TAVILY_API_KEY: [LOADED]


[Here](https://python.langchain.com/v0.2/docs/how_to/#chat-models) is a useful how-to for all the things that you can do with chat models, but we'll show a few highlights below. If you've run `pip install -r requirements.txt` as noted in the README, then you've installed the `langchain-openai` package. With this, we can instantiate our `ChatOpenAI` model object. If you are signing up for the API for the first time, you should receive [free credits](https://community.openai.com/t/understanding-api-limits-and-free-tier/498517) that can be applied to any of the models. You can see pricing for various models [here](https://openai.com/api/pricing/). The notebooks will default to `gpt-4o` because it's a good balance of quality, price, and speed [see more here](https://help.openai.com/en/articles/7102672-how-can-i-access-gpt-4-gpt-4-turbo-gpt-4o-and-gpt-4o-mini), but you can also opt for the lower priced `gpt-3.5` series models. 

There are [a few standard parameters](https://python.langchain.com/v0.2/docs/concepts/#chat-models) that we can set with chat models. Two of the most common are:

* `model`: the name of the model
* `temperature`: the sampling temperature

`Temperature` controls the randomness or creativity of the model's output where low temperature (close to 0) is more deterministic and focused outputs. This is good for tasks requiring accuracy or factual responses. High temperature (close to 1) is good for creative tasks or generating varied responses. 

In [5]:
from langchain_openai import ChatOpenAI
gpt4o_chat = ChatOpenAI(model="gpt-4o", temperature=0)
gpt35_chat = ChatOpenAI(model="gpt-3.5-turbo-0125", temperature=0)

Chat models in LangChain have a number of [default methods](https://python.langchain.com/v0.2/docs/concepts/#runnable-interface). For the most part, we'll be using:

* `stream`: stream back chunks of the response
* `invoke`: call the chain on an input

And, as mentioned, chat models take [messages](https://python.langchain.com/v0.2/docs/concepts/#messages) as input. Messages have a role (that describes who is saying the message) and a content property. We'll be talking a lot more about this later, but here let's just show the basics.

In [6]:
from langchain_core.messages import HumanMessage

# Create a message
msg = HumanMessage(content="Hello world", name="Lance")

# Message list
messages = [msg]

# Invoke the model with a list of messages 
gpt4o_chat.invoke(messages)

AIMessage(content='Hello! How can I assist you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 9, 'prompt_tokens': 11, 'total_tokens': 20, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_126ee87271', 'id': 'chatcmpl-DMrpq0yyhCWBDXUR62hShvIsZ5tEJ', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d1f0e-6115-7da2-a4c9-31d7d15c1e5f-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 11, 'output_tokens': 9, 'total_tokens': 20, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

## Search Tools

You'll also see [Tavily](https://tavily.com/) in the README, which is a search engine optimized for LLMs and RAG, aimed at efficient, quick, and persistent search results. As mentioned, it's easy to sign up and offers a generous free tier. Some lessons (in Module 4) will use Tavily by default but, of course, other search tools can be used if you want to modify the code for yourself.

In [12]:
print("TAVILY_API_KEY:", "[LOADED]" if os.getenv("TAVILY_API_KEY") else "Not found")

TAVILY_API_KEY: [LOADED]


In [16]:
from langchain_community.tools.tavily_search import TavilySearchResults
tavily_search = TavilySearchResults(max_results=3)
search_docs = tavily_search.invoke("Tell me few lines about SR University, Warangal, Telengana, India")


[{'title': 'Best University in Telangana for Engineering & More', 'url': 'https://sru.edu.in/', 'content': 'Testimonial, SR University\n\n- Savant Arthi, Dept. of ECE (2010-14) Senior Associate Consultant, Infosys\n\nTestimonial, SR University\n\n- Ravula Shirisha,  Dept. of CSE (2015-19) Project Engineer, Wipro\n\n##### Enquire Now\n\n##### Admissions\n\n##### Admissions\n\n##### Other Links:\n\n##### Other Links\n\n#### \n\n##### Media\n\n##### \n\n##### Quick Connects\n\n##### Quick Connects\n\n###### Visit Us\n\n###### Academics\n\n###### Campus life\n\n###### Admissions\n\n###### Exam Branch\n\n###### AC-Holidays List\n\n###### Careers\n\n###### UGC\n\n###### NIRF\n\n###### University Library\n\nAddress icon png, Location icon png, SR University\n\n##### SR UNIVERSITY\n\nAnanthasagar, Hasanparthy\n\nWarangal - 506371, Telangana, India\n\nMail icon png, Telephone icon png, SR University\n\n0(870) 281-8333/8311\n\ninfo@sru.edu.in [...] 0(870) 281-8333/8311\n\ninfo@sru.edu.in\n\n####

In [19]:
from pprint import pprint
pprint(search_docs)

[{'content': 'Testimonial, SR University\n'
             '\n'
             '- Savant Arthi, Dept. of ECE (2010-14) Senior Associate '
             'Consultant, Infosys\n'
             '\n'
             'Testimonial, SR University\n'
             '\n'
             '- Ravula Shirisha,  Dept. of CSE (2015-19) Project Engineer, '
             'Wipro\n'
             '\n'
             '##### Enquire Now\n'
             '\n'
             '##### Admissions\n'
             '\n'
             '##### Admissions\n'
             '\n'
             '##### Other Links:\n'
             '\n'
             '##### Other Links\n'
             '\n'
             '#### \n'
             '\n'
             '##### Media\n'
             '\n'
             '##### \n'
             '\n'
             '##### Quick Connects\n'
             '\n'
             '##### Quick Connects\n'
             '\n'
             '###### Visit Us\n'
             '\n'
             '###### Academics\n'
             '\n'
             '###### C